In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv('C:/Users/human/미정프로젝트/서울시 일반음식점 인허가 정보.csv',
                  encoding='cp949', encoding_errors='replace', low_memory=False)
df2 = pd.read_csv('C:/Users/human/미정프로젝트/서울시 휴게음식점 인허가 정보.csv',
                  encoding='cp949', encoding_errors='replace', low_memory=False)

df1['업종구분'] = '일반음식점'
df2['업종구분'] = '휴게음식점'
df = pd.concat([df1, df2], ignore_index=True)
print(f'합친 행 수: {len(df):,}')

합친 행 수: 677,839


In [3]:
cols = ['사업장명', '업태구분명', '업종구분', '영업상태명',
        '인허가일자', '폐업일자', '도로명주소']
df = df[cols].copy()

df['인허가일자'] = pd.to_datetime(df['인허가일자'], errors='coerce')
df['폐업일자'] = pd.to_datetime(df['폐업일자'], errors='coerce')
df['구명'] = df['도로명주소'].str.extract(r'서울특별시\s+(\S+구)')
df_seoul = df[df['구명'].notna()].copy()

print(f'서울 데이터: {len(df_seoul):,}개')

서울 데이터: 386,214개


In [4]:
업종_매핑 = {
    '한식': '한식음식점',
    '커피숍': '커피-음료',
    '경양식': '양식음식점',
    '일식': '일식음식점',
    '중국식': '중식음식점',
    '호프/통닭': '치킨전문점',
    '분식': '분식전문점',
}

df_food = df_seoul[df_seoul['업태구분명'].isin(업종_매핑.keys())].copy()
df_food['업종'] = df_food['업태구분명'].map(업종_매핑)

print(f'필터링 후: {len(df_food):,}행')
print(df_food['업종'].value_counts())

필터링 후: 241,425행
업종
한식음식점    114170
커피-음료     37634
치킨전문점     23574
양식음식점     21689
분식전문점     20012
일식음식점     14384
중식음식점      9962
Name: count, dtype: int64


In [7]:
results = []

for year in range(2019, 2025):
    영업중 = df_food[
        (df_food['인허가일자'].dt.year <= year) &
        ((df_food['폐업일자'].dt.year > year) | (df_food['폐업일자'].isna()))
    ].copy()
    영업중['연도'] = year
    영업중['해당연도_폐업'] = 0

    폐업 = df_food[df_food['폐업일자'].dt.year == year].copy()
    폐업['연도'] = year
    폐업['해당연도_폐업'] = 1

    results.append(영업중)
    results.append(폐업)

df_yearly = pd.concat(results, ignore_index=True)

closure_detail = df_yearly.groupby(['구명', '연도', '업종']).agg(
    총개수=('해당연도_폐업', 'count'),
    폐업수=('해당연도_폐업', 'sum')
).reset_index()

closure_detail['폐업률'] = (closure_detail['폐업수'] / closure_detail['총개수'] * 100).round(2)
closure_detail = closure_detail[closure_detail['총개수'] >= 5]

print(f'총 조합 수: {len(closure_detail):,}개')
print(closure_detail.head(10))

총 조합 수: 1,050개
    구명    연도     업종   총개수  폐업수    폐업률
0  강남구  2019  분식전문점   997   78   7.82
1  강남구  2019  양식음식점  2516  249   9.90
2  강남구  2019  일식음식점   938   96  10.23
3  강남구  2019  중식음식점   402   24   5.97
4  강남구  2019  치킨전문점   261   32  12.26
5  강남구  2019  커피-음료  1564  330  21.10
6  강남구  2019  한식음식점  5277  449   8.51
7  강남구  2020  분식전문점  1012   81   8.00
8  강남구  2020  양식음식점  2512  265  10.55
9  강남구  2020  일식음식점   955   94   9.84


In [9]:
closure_detail.to_csv('폐업률_구별_연도별_업종별.csv', index=False, encoding='utf-8-sig')
print('저장 완료!')
print(closure_detail['폐업률'].describe().round(2))

저장 완료!
count    1050.00
mean       10.02
std         3.11
min         1.98
25%         8.03
50%         9.75
75%        11.77
max        23.76
Name: 폐업률, dtype: float64


In [12]:
results_comp = []

for year in range(2019, 2025):
    영업중 = df_food[
        (df_food['인허가일자'].dt.year <= year) &
        ((df_food['폐업일자'].dt.year > year) | (df_food['폐업일자'].isna()))
    ].copy()
    영업중['연도'] = year
    results_comp.append(영업중)

df_comp = pd.concat(results_comp, ignore_index=True)

df_competition = df_comp.groupby(['구명', '연도', '업종']).agg(
    경쟁강도=('사업장명', 'count')
).reset_index()

print(f'행 수: {len(df_competition):,}개')
print(df_competition.head(10))

행 수: 1,050개
    구명    연도     업종  경쟁강도
0  강남구  2019  분식전문점   919
1  강남구  2019  양식음식점  2267
2  강남구  2019  일식음식점   842
3  강남구  2019  중식음식점   378
4  강남구  2019  치킨전문점   229
5  강남구  2019  커피-음료  1234
6  강남구  2019  한식음식점  4828
7  강남구  2020  분식전문점   931
8  강남구  2020  양식음식점  2247
9  강남구  2020  일식음식점   861


In [13]:
# ML 데이터에 경쟁강도 추가
df_ml = pd.read_csv('C:/Users/human/미정프로젝트/새 폴더/최종_ML용_데이터_v2.csv',
                    encoding='utf-8-sig')

df_ml = pd.merge(df_ml, df_competition, on=['구명', '연도', '업종'], how='left')

print(f'행 수: {len(df_ml):,}개')
print(f'결측치: {df_ml["경쟁강도"].isnull().sum()}개')
print(df_ml[['구명', '연도', '업종', '폐업률', '임대료', '경쟁강도']].head())

# 저장
df_ml.to_csv('C:/Users/human/미정프로젝트/새 폴더/최종_ML용_데이터_v2.csv',
             index=False, encoding='utf-8-sig')
print('\n저장 완료!')

행 수: 1,050개
결측치: 0개
    구명    연도     업종    폐업률     임대료  경쟁강도
0  강남구  2019  분식전문점   7.82  130580   919
1  강남구  2019  양식음식점   9.90  130580  2267
2  강남구  2019  일식음식점  10.23  130580   842
3  강남구  2019  중식음식점   5.97  130580   378
4  강남구  2019  치킨전문점  12.26  130580   229

저장 완료!


In [18]:
import pandas as pd

# 파일 읽기
df1 = pd.read_csv('C:/Users/human/미정프로젝트/서울시 일반음식점 인허가 정보.csv',
                  encoding='cp949', encoding_errors='replace', low_memory=False)
df2 = pd.read_csv('C:/Users/human/미정프로젝트/서울시 휴게음식점 인허가 정보.csv',
                  encoding='cp949', encoding_errors='replace', low_memory=False)

df1['업종구분'] = '일반음식점'
df2['업종구분'] = '휴게음식점'
df = pd.concat([df1, df2], ignore_index=True)

cols = ['사업장명', '업태구분명', '업종구분', '영업상태명',
        '인허가일자', '폐업일자', '도로명주소']
df = df[cols].copy()

df['인허가일자'] = pd.to_datetime(df['인허가일자'], errors='coerce')
df['폐업일자'] = pd.to_datetime(df['폐업일자'], errors='coerce')
df['구명'] = df['도로명주소'].str.extract(r'서울특별시\s+(\S+구)')
df_seoul = df[df['구명'].notna()].copy()

업종_매핑 = {
    '한식': '한식음식점', '커피숍': '커피-음료',
    '경양식': '양식음식점', '일식': '일식음식점',
    '중국식': '중식음식점', '호프/통닭': '치킨전문점',
    '분식': '분식전문점',
}
df_food = df_seoul[df_seoul['업태구분명'].isin(업종_매핑.keys())].copy()
df_food['업종'] = df_food['업태구분명'].map(업종_매핑)
df_food['폐업여부'] = (df_food['영업상태명'] == '폐업').astype(int)

print(f'df_food: {len(df_food):,}행')
print(df_food.columns.tolist())

df_food: 241,425행
['사업장명', '업태구분명', '업종구분', '영업상태명', '인허가일자', '폐업일자', '도로명주소', '구명', '업종', '폐업여부']


In [20]:
# df_closed_all 만들기
df_closed_all = df_food[df_food['폐업여부'] == 1].copy()
df_closed_all['생존_년수'] = ((df_closed_all['폐업일자'] - df_closed_all['인허가일자']).dt.days / 365).round(1)

# 강남구만
df_closed_gangnam = df_closed_all[df_closed_all['구명'] == '강남구']
print(df_closed_gangnam.groupby('업종')['생존_년수'].mean().sort_values())

업종
커피-음료    2.107454
치킨전문점    4.942199
일식음식점    6.436041
한식음식점    7.738028
양식음식점    8.043370
중식음식점    8.251030
분식전문점    9.350190
Name: 생존_년수, dtype: float64


In [21]:
df_closed = df[df['폐업률'] > 0].copy()
print(df[df['구명'] == '강남구'].groupby('업종')['폐업률'].mean().sort_values(ascending=False))

KeyError: '폐업률'

In [22]:
df_closed_gangnam = df_closed_all[df_closed_all['구명'] == '강남구']
df_closed_gangnam['생존_년수'] = ((df_closed_gangnam['폐업일자'] - df_closed_gangnam['인허가일자']).dt.days / 365).round(1)

print(df_closed_gangnam.groupby('업종')['생존_년수'].mean().sort_values())

업종
커피-음료    2.107454
치킨전문점    4.942199
일식음식점    6.436041
한식음식점    7.738028
양식음식점    8.043370
중식음식점    8.251030
분식전문점    9.350190
Name: 생존_년수, dtype: float64


C:\Users\human\AppData\Local\Temp\ipykernel_7968\2364061806.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_closed_gangnam['생존_년수'] = ((df_closed_gangnam['폐업일자'] - df_closed_gangnam['인허가일자']).dt.days / 365).round(1)


In [23]:
print('=== 서울 전체 평균 생존 년수 ===')
df_closed_all['생존_년수'] = ((df_closed_all['폐업일자'] - df_closed_all['인허가일자']).dt.days / 365).round(1)
print(df_closed_all.groupby('업종')['생존_년수'].mean().sort_values())

=== 서울 전체 평균 생존 년수 ===
업종
커피-음료    3.859809
일식음식점    6.883022
양식음식점    7.960406
한식음식점    8.700167
치킨전문점    8.818467
중식음식점    9.580845
분식전문점    9.721500
Name: 생존_년수, dtype: float64


In [24]:
df_closed_gangnam = df_closed_all[df_closed_all['구명'] == '강남구'].copy()
df_closed_gangnam['생존_년수'] = ((df_closed_gangnam['폐업일자'] - df_closed_gangnam['인허가일자']).dt.days / 365).round(1)

서울_평균 = df_closed_all.groupby('업종')['생존_년수'].mean().round(1)
강남_평균 = df_closed_gangnam.groupby('업종')['생존_년수'].mean().round(1)

df_compare = pd.DataFrame({
    '서울_평균': 서울_평균,
    '강남구': 강남_평균
})
df_compare['차이'] = (df_compare['강남구'] - df_compare['서울_평균']).round(1)
df_compare = df_compare.sort_values('차이')

print(df_compare)

       서울_평균  강남구   차이
업종                    
치킨전문점    8.8  4.9 -3.9
커피-음료    3.9  2.1 -1.8
중식음식점    9.6  8.3 -1.3
한식음식점    8.7  7.7 -1.0
일식음식점    6.9  6.4 -0.5
분식전문점    9.7  9.4 -0.3
양식음식점    8.0  8.0  0.0
